In [7]:
import sys
from pathlib import Path

# Add parent directory to path so we can import from src
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from src.agent import create_agent, PictionaryAgent
from PIL import Image
import numpy as np

# PictionaryAgent Examples

Shows how to use the `PictionaryAgent` for sketch recognition with direct inference and policy-based guessing.

In [10]:
## Setup: Load Test Image
# You won't have to actually do this for the app,
# just pass in a PIL image to the agent and it will handle the rest
# actively ignore this code please, it's just to let me show the guessing methods below.
from src.data import DataConfig, ALL_QUICKDRAW_CATEGORIES

# I'm slicing to :50 because we only trianed on 50 categories,
# I'm not changing that ATP... 
config = DataConfig(categories=ALL_QUICKDRAW_CATEGORIES[:50], max_per_class=5, 
                    train_split=0.8, batch_size=1, num_workers=0)
_, val_loader = config.get_loaders()

# Get sample and convert to PIL Image
image_tensor, label = val_loader.dataset[0]
true_category = config.categories[label]
img_array = ((image_tensor.squeeze().numpy() + 1.0) * 127.5).astype(np.uint8)
pil_image = Image.fromarray(img_array, mode='L')

print(f"True category: {true_category}")
pil_image

✓ aircraft carrier already downloaded
✓ airplane already downloaded
✓ alarm clock already downloaded
✓ ambulance already downloaded
✓ angel already downloaded
✓ animal migration already downloaded
✓ ant already downloaded
✓ anvil already downloaded
✓ apple already downloaded
✓ arm already downloaded
✓ asparagus already downloaded
✓ axe already downloaded
✓ backpack already downloaded
✓ banana already downloaded
✓ bandage already downloaded
✓ barn already downloaded
✓ baseball already downloaded
✓ baseball bat already downloaded
✓ basket already downloaded
✓ basketball already downloaded
✓ bat already downloaded
✓ bathtub already downloaded
✓ beach already downloaded
✓ bear already downloaded
✓ beard already downloaded
✓ bed already downloaded
✓ bee already downloaded
✓ belt already downloaded
✓ bench already downloaded
✓ bicycle already downloaded
✓ binoculars already downloaded
✓ bird already downloaded
✓ birthday cake already downloaded
✓ blackberry already downloaded
✓ blueberry alr

/var/folders/b9/cpb3kh2n0lv70gkm_76yfznw0000gn/T/ipykernel_39299/2973038124.py:17: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil_image = Image.fromarray(img_array, mode='L')


In [15]:
# Create agent gets you an agent with the model loaded
# Seting use_policy=False means the agent will ignore any guessing policies and 
# just return the raw model predictions
agent = create_agent('resnet18', models_dir=str(project_root / 'models'), use_policy=False)
# Predict, just uses the base model, ignoring any policies, idk how we want
# the frontend to look ,but this is how you get a guess from the base model without any policy logic
# We can retrun the top_k guesses if we want.
result = agent.predict(pil_image, return_top_k=3)
print(result)
print(f"Top guess: {result['top_guess']} ({result['top_confidence']:.1%})")
print(f"Top 3: {', '.join([g['label'] for g in result['guesses']])}")

{'top_guess': 'apple', 'top_confidence': 0.9995743632316589, 'guesses': [{'label': 'apple', 'confidence': 0.9995743632316589}, {'label': 'blueberry', 'confidence': 0.00042313901940360665}, {'label': 'broccoli', 'confidence': 1.3581690154751414e-06}], 'all_logits': tensor([[ -9.3141,  -8.6513,  -5.4145,  -9.8700,  -5.9174,  -5.4471,  -9.3900,
          -5.8096,  12.6507,  -6.8656,  -1.9201,  -7.4389,  -8.7541,  -3.7018,
         -10.4202,  -9.1249, -10.5834, -11.2304,  -6.7800,  -7.1449,  -7.8485,
          -9.1821,  -7.4974,  -4.6907,  -4.9418,  -8.8618,  -6.3773,  -5.0986,
          -8.4890,  -8.0806,  -7.9105,  -7.2072,  -7.1093,  -1.9962,   4.8833,
         -10.8176,  -9.0402,  -7.3179,  -7.2163,  -4.8239,  -8.4637,  -7.7783,
          -8.9129,  -0.8583,  -5.9655,  -9.4289,  -6.5299,  -8.6085,  -9.8192,
          -3.8020]])}
Top guess: apple (100.0%)
Top 3: apple, blueberry, broccoli


## Policy-Based Guessing

Policies decide when to commit to a guess. Use `predict_with_policy()` with `num_strokes` parameter.

In [ ]:
# this code shows how to use the different policies we trained,
# you can swap out the policy_type to see how each one behaves on the same image
# Confidence policy: guesses when confident enough
conf_agent = create_agent('resnet18', models_dir=str(project_root / 'models'), 
                          policies_dir=str(project_root / 'models/trained_policies'),
                          use_policy=True, policy_type='confidence')
conf_agent.reset()
# predict_with_policy returns a dict with 'should_guess', 'top_guess', and 'top_confidence' keys
# should_guess is the policy's decision on whether to guess or wait for more strokes
# top_guess and top_confidence are the model's current best guess and its confidence, 
# which the policy uses to make its decision
# You NEED to pass a num_strokes argument, as the time based policy needs it to decide
# when to guess. I think the app already tracks the number of strokes, so just pass that in
result = conf_agent.predict_with_policy(pil_image, num_strokes=1)
print(f"Confidence policy - Should guess: {result['should_guess']}")
if result['should_guess']:
    print(f"  Guess: {result['top_guess']} ({result['top_confidence']:.1%})")

# Time policy: guesses after N strokes
time_agent = create_agent('vit', models_dir=str(project_root / 'models'),
                         policies_dir=str(project_root / 'models/trained_policies'),
                         use_policy=True, policy_type='time')
time_agent.reset()
result = time_agent.predict_with_policy(pil_image, num_strokes=1)
print(f"\nTime policy - Should guess: {result['should_guess']}")
if result['should_guess']:
    print(f"  Guess: {result['top_guess']} ({result['top_confidence']:.1%})")

# Learned policy: neural network decides
learned_agent = create_agent('mlp', models_dir=str(project_root / 'models'),
                            policies_dir=str(project_root / 'models/trained_policies'),
                            use_policy=True, policy_type='learned')
learned_agent.reset()
result = learned_agent.predict_with_policy(pil_image, num_strokes=1)
print(f"\nLearned policy - Should guess: {result['should_guess']}")
if result['should_guess']:
    correct = "✓" if result['top_guess'] == true_category else "✗"
    print(f"  Guess: {result['top_guess']} ({result['top_confidence']:.1%}) {correct}")

Confidence policy - Should guess: True
  Guess: apple (100.0%)

Time policy - Should guess: False

Learned policy - Should guess: False


## Compare All Models

In [ ]:
models_dir = str(project_root / 'models')
agents = {name: create_agent(name, models_dir=models_dir, use_policy=False) 
          for name in ['mlp', 'resnet18', 'vit']}

for name, agent in agents.items():
    result = agent.predict(pil_image, return_top_k=1)
    correct = "✓" if result['top_guess'] == true_category else "✗"
    print(f"{name.upper():8} {correct} {result['top_guess']:20} ({result['top_confidence']:.1%})")

MLP      ✓ apple                (73.1%)
RESNET18 ✓ apple                (100.0%)
VIT      ✓ apple                (99.8%)
